# 🎧 Music Recommender | SQL Edition

## 05. Limpieza de Spotify Tracks 114K

Este notebook aplica las reglas de limpieza derivadas de `01_spotify_114k_exploration.ipynb`. Partimos del CSV original, con 114.000 filas y 21 columnas.

La unidad de trabajo es la asociación entre una canción y un género. Una canción puede aparecer en varios géneros, por lo que conservaremos un registro por cada par `track_id`–`track_genre`. Retiraremos el índice de exportación y las copias redundantes. También documentaremos la exclusión del registro con duración cero y sin artista, álbum ni título.

Conservaremos los valores de popularidad observados por asociación, las etiquetas de género originales y las grabaciones con valores acústicos atípicos. Al final validaremos los recuentos y la integridad de los datos antes de guardar el conjunto limpio.

In [2]:
from pathlib import Path

import pandas as pd

ruta_original = Path("../data/raw/spotify_tracks_114k.csv")
spotify_raw = pd.read_csv(ruta_original)

filas, columnas = spotify_raw.shape
print(f"Filas cargadas: {filas}")
print(f"Columnas cargadas: {columnas}")

Filas cargadas: 114000
Columnas cargadas: 21


In [3]:
# Comprobar identificadores repetidos
print("Filas totales:", len(spotify_raw))
print("Track IDs únicos:", spotify_raw["track_id"].nunique())
print("Filas con track_id repetido:", spotify_raw["track_id"].duplicated().sum())

# Mostrar ejemplos de canciones con identificadores repetidos
duplicados = spotify_raw[
    spotify_raw["track_id"].duplicated(keep=False)
]

display(duplicados.sort_values("track_id").head(10))

Filas totales: 114000
Track IDs únicos: 89741
Filas con track_id repetido: 24259


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
15028,15028,001APMDOl3qtx1526T11n1,Pink Sweat$;Kirby,New RnB,Better,0,176320,False,0.613,0.471,...,-6.644,0,0.1070,0.31600,0.000001,0.1170,0.406,143.064,4,chill
103211,103211,001APMDOl3qtx1526T11n1,Pink Sweat$;Kirby,New RnB,Better,0,176320,False,0.613,0.471,...,-6.644,0,0.1070,0.31600,0.000001,0.1170,0.406,143.064,4,soul
85578,85578,001YQlnDSduXd5LgBd66gT,Soda Stereo,Soda Stereo (Remastered),El Tiempo Es Dinero - Remasterizado 2007,38,177266,False,0.554,0.921,...,-4.589,1,0.0758,0.01940,0.088100,0.3290,0.700,183.571,1,punk-rock
100420,100420,001YQlnDSduXd5LgBd66gT,Soda Stereo,Soda Stereo (Remastered),El Tiempo Es Dinero - Remasterizado 2007,38,177266,False,0.554,0.921,...,-4.589,1,0.0758,0.01940,0.088100,0.3290,0.700,183.571,1,ska
91801,91801,003vvx7Niy0yvhvHt4a68B,The Killers,Hot Fuss,Mr. Brightside,86,222973,False,0.352,0.911,...,-5.230,1,0.0747,0.00121,0.000000,0.0995,0.236,148.033,4,rock
3257,3257,003vvx7Niy0yvhvHt4a68B,The Killers,Hot Fuss,Mr. Brightside,86,222973,False,0.352,0.911,...,-5.230,1,0.0747,0.00121,0.000000,0.0995,0.236,148.033,4,alternative
2106,2106,003vvx7Niy0yvhvHt4a68B,The Killers,Hot Fuss,Mr. Brightside,86,222973,False,0.352,0.911,...,-5.230,1,0.0747,0.00121,0.000000,0.0995,0.236,148.033,4,alt-rock
33178,33178,004h8smbIoAkUNDJvVKwkG,Ouse;Powfu,Loners Diary,Lovemark,58,219482,True,0.808,0.331,...,-13.457,1,0.0557,0.13100,0.000000,0.2250,0.337,140.035,4,emo
94239,94239,004h8smbIoAkUNDJvVKwkG,Ouse;Powfu,Loners Diary,Lovemark,58,219482,True,0.808,0.331,...,-13.457,1,0.0557,0.13100,0.000000,0.2250,0.337,140.035,4,sad
97533,97533,006rHBBNLJMpQs8fRC2GDe,Calcinha Preta;Gusttavo Lima,CP 25 Anos (Ao Vivo em Aracaju),Agora Estou Sofrendo - Ao Vivo,47,260510,False,0.605,0.678,...,-3.257,1,0.0311,0.64200,0.000000,0.1570,0.439,125.059,4,sertanejo


In [4]:
# Comparar registros ignorando el género y el índice original
columnas_comparacion = spotify_raw.columns.difference(
    ["Unnamed: 0", "track_genre"]
)

variaciones = (
    spotify_raw
    .groupby("track_id")[columnas_comparacion]
    .nunique(dropna=False)
)

ids_con_diferencias = variaciones.gt(1).any(axis=1).sum()

print("Track IDs con diferencias adicionales al género:", ids_con_diferencias)

Track IDs con diferencias adicionales al género: 720


In [5]:
# Contar cuántos track_id presentan diferencias en cada columna
columnas_con_diferencias = (
    variaciones.gt(1)
    .sum()
    .sort_values(ascending=False)
)

display(
    columnas_con_diferencias[
        columnas_con_diferencias > 0
    ]
)


popularity    720
dtype: int64

In [6]:
# Analizar las diferencias de popularidad por canción
popularidad = spotify_raw.groupby("track_id")["popularity"].agg(
    ["min", "max"]
)

popularidad["diferencia"] = (
    popularidad["max"] - popularidad["min"]
)

# Mostrar únicamente las canciones con diferencias
popularidad_variable = popularidad[
    popularidad["diferencia"] > 0
]

print("Canciones con popularidad variable:", len(popularidad_variable))

display(
    popularidad_variable.sort_values(
        "diferencia", ascending=False
    ).head(10)
)

Canciones con popularidad variable: 720


,min,max,diferencia
track_id,,,
5hljqZrl0h7RcIhUnOPuve,0,44,44
1UwUhKmFxGKs59xiWO60Sx,0,42,42
0C6dSPCqwKqwos6s1CrDrh,0,42,42
5ygEUpyZy5qtZ1423zymBW,0,41,41
1zx6GSqLYI2ynzAHnPRKBR,0,41,41
4KPi4VNvPOfKqIp1sx19Xf,0,41,41
5fClRRa7P9nPIpstj50veV,0,40,40
0z5x3VmmQ7cyTeOP7VAf3T,0,40,40
42GuKw49pPxNAkIhWGwgFs,0,39,39


In [7]:
# Investigar los valores de popularidad diferentes

print(
    "Canciones con mínimo igual a 0:",
    (popularidad_variable["min"] == 0).sum()
)

print(
    "Canciones con mínimo mayor que 0:",
    (popularidad_variable["min"] > 0).sum()
)

print(
    "Diferencia media de popularidad:",
    round(popularidad_variable["diferencia"].mean(), 2)
)

print(
    "Diferencia mediana de popularidad:",
    popularidad_variable["diferencia"].median()
)

Canciones con mínimo igual a 0: 105
Canciones con mínimo mayor que 0: 615
Diferencia media de popularidad: 1.71
Diferencia mediana de popularidad: 1.0


In [8]:
# Comprobar identificadores de canciones

print(
    "Track IDs nulos:",
    spotify_raw["track_id"].isna().sum()
)

print(
    "Track IDs vacíos:",
    spotify_raw["track_id"].fillna("").str.strip().eq("").sum()
)

Track IDs nulos: 0
Track IDs vacíos: 0


In [9]:
# Crear una copia de los datos originales
spotify_clean = spotify_raw.copy()

# Eliminar la columna de índice innecesaria
spotify_clean = spotify_clean.drop(columns=["Unnamed: 0"])

print("Filas:", spotify_clean.shape[0])
print("Columnas:", spotify_clean.shape[1])

display(spotify_clean.head())

Filas: 114000
Columnas: 20


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [10]:
# Crear la tabla de relaciones entre canciones y géneros
track_genres = (
    spotify_clean[["track_id", "track_genre"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Relaciones canción-género:", len(track_genres))
print("Canciones únicas:", track_genres["track_id"].nunique())
print("Géneros únicos:", track_genres["track_genre"].nunique())

display(track_genres.head(10))

Relaciones canción-género: 113550
Canciones únicas: 89741
Géneros únicos: 114


,track_id,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,acoustic
4,5vjLSffimiIP26QG5WcN2K,acoustic
5,01MVOl9KtVTNfFiBU9I7dc,acoustic
6,6Vc5wAMmXdKIAM7WUoEb7N,acoustic
7,1EzrEOXmMH3G43AXT1y7pA,acoustic
8,0IktbUcnAGrvD03AWnz3Q8,acoustic
9,7k9GuJYLp2AzqokyEdwEw2,acoustic


In [11]:
# Crear una tabla con una fila por canción
tracks = (
    spotify_clean
    .sort_values("popularity", ascending=False)
    .drop_duplicates(subset="track_id", keep="first")
    .drop(columns=["track_genre"])
    .reset_index(drop=True)
)

print("Canciones en tracks:", len(tracks))
print("Track IDs únicos:", tracks["track_id"].nunique())
print("Duplicados por track_id:", tracks["track_id"].duplicated().sum())

display(tracks.head())

Canciones en tracks: 89741
Track IDs únicos: 89741
Duplicados por track_id: 0


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,3nqQXoyQOWXiESFLlDF1hG,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),Unholy (feat. Kim Petras),100,156943,False,0.714,0.472,2,-7.375,1,0.0864,0.01300,0.000005,0.2660,0.238,131.121,4
1,2tTmW7RDtMQtBk7m2rYeSw,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52","Quevedo: Bzrp Music Sessions, Vol. 52",99,198937,False,0.621,0.782,2,-5.548,1,0.0440,0.01250,0.033000,0.2300,0.550,128.033,4
2,5ww2BF9slyYgNOk37BlC4u,Manuel Turizo,La Bachata,La Bachata,98,162637,False,0.835,0.679,7,-5.329,0,0.0364,0.58300,0.000002,0.2180,0.850,124.980,4
3,4uUG5RXrOk84mYEfFvj3cK,David Guetta;Bebe Rexha,I'm Good (Blue),I'm Good (Blue),98,175238,True,0.561,0.965,7,-3.673,0,0.0343,0.00383,0.000007,0.3710,0.304,128.040,4
4,6Sq7ltF9Qa7SNFBsV5Cogx,Bad Bunny;Chencho Corleone,Un Verano Sin Ti,Me Porto Bonito,97,178567,True,0.911,0.712,1,-5.105,0,0.0817,0.09010,0.000027,0.0933,0.425,92.005,4


In [12]:
# Comprobar que todos los géneros pertenecen a canciones existentes

ids_tracks = set(tracks["track_id"])

relaciones_huerfanas = track_genres[
    ~track_genres["track_id"].isin(ids_tracks)
]

print("Relaciones canción-género:", len(track_genres))
print("Relaciones sin canción asociada:", len(relaciones_huerfanas))

Relaciones canción-género: 113550
Relaciones sin canción asociada: 0


In [13]:
# Comprobar valores nulos en las tablas preparadas

print("Valores nulos en tracks:")
display(tracks.isna().sum().sort_values(ascending=False))

print("\nValores nulos en track_genres:")
display(track_genres.isna().sum())

Valores nulos en tracks:


artists             1
album_name          1
track_name          1
track_id            0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
dtype: int64


Valores nulos en track_genres:


track_id       0
track_genre    0
dtype: int64

In [14]:
# Localizar canciones con información descriptiva incompleta

columnas_texto = ["artists", "album_name", "track_name"]

tracks_incompletos = tracks[
    tracks[columnas_texto].isna().any(axis=1)
]

print("Canciones con datos incompletos:", len(tracks_incompletos))

display(
    tracks_incompletos[
        ["track_id", "artists", "album_name",
         "track_name", "popularity"]
    ]
)

Canciones con datos incompletos: 1


,track_id,artists,album_name,track_name,popularity
82989,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0


In [15]:
# Buscar todas las apariciones del registro incompleto
track_id_incompleto = tracks_incompletos.iloc[0]["track_id"]

registro_original = spotify_raw[
    spotify_raw["track_id"] == track_id_incompleto
]

print("Apariciones en el dataset original:", len(registro_original))

display(
    registro_original[
        ["track_id", "artists", "album_name",
         "track_name", "popularity", "track_genre"]
    ]
)

Apariciones en el dataset original: 1


,track_id,artists,album_name,track_name,popularity,track_genre
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,k-pop


In [16]:
# Excluir la canción sin información descriptiva
tracks = tracks[
    tracks["track_id"] != track_id_incompleto
].reset_index(drop=True)

# Eliminar también su relación con el género
track_genres = track_genres[
    track_genres["track_id"] != track_id_incompleto
].reset_index(drop=True)

# Comprobar los resultados
print("Canciones restantes:", len(tracks))
print("Relaciones canción-género:", len(track_genres))
print("Valores nulos en tracks:", tracks.isna().sum().sum())
print("Valores nulos en track_genres:", track_genres.isna().sum().sum())

Canciones restantes: 89740
Relaciones canción-género: 113549
Valores nulos en tracks: 0
Valores nulos en track_genres: 0


In [17]:
# Validar las características musicales entre 0 y 1

columnas_audio = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence"
]

for columna in columnas_audio:
    fuera_rango = (~tracks[columna].between(0, 1)).sum()
    print(f"{columna}: {fuera_rango} valores fuera de rango")

danceability: 0 valores fuera de rango
energy: 0 valores fuera de rango
speechiness: 0 valores fuera de rango
acousticness: 0 valores fuera de rango
instrumentalness: 0 valores fuera de rango
liveness: 0 valores fuera de rango
valence: 0 valores fuera de rango


In [18]:
# Revisar los rangos de las demás variables numéricas

columnas_numericas = [
    "popularity",
    "duration_ms",
    "tempo",
    "loudness",
    "key",
    "mode",
    "time_signature"
]

display(
    tracks[columnas_numericas]
    .agg(["min", "max"])
    .T
)

,min,max
popularity,0.000,100.000
duration_ms,8586.000,5237295.000
tempo,0.000,243.372
loudness,-49.531,4.532
key,0.000,11.000
mode,0.000,1.000
time_signature,0.000,5.000


## Resultados de la limpieza inicial | Pasos 1-17

### 1. Carga y preparación del dataset

Se ha trabajado con el dataset Spotify Tracks 114K, compuesto originalmente por **114.000 registros y 21 columnas**.

Se ha creado una copia de trabajo (`spotify_clean`) para preservar los datos originales y se ha eliminado la columna `Unnamed: 0`, correspondiente a un índice anterior.

### 2. Análisis de duplicados

Se han identificado:

- 89.741 identificadores de canciones únicos.
- 24.259 apariciones adicionales de identificadores repetidos.
- 720 canciones con diferencias en la columna `popularity`.

Las diferencias entre registros de un mismo `track_id` se concentran en el género musical y, en determinados casos, en la popularidad.

Para consolidar las canciones se ha seleccionado provisionalmente la popularidad máxima de cada identificador.

### 3. Separación de canciones y géneros

Se han creado dos DataFrames:

- `tracks`: una fila por canción.
- `track_genres`: relaciones únicas entre canciones y géneros.

Inicialmente se han obtenido 89.741 canciones y 113.550 relaciones canción-género, correspondientes a 114 géneros diferentes.

Esta separación permite preparar una relación muchos a muchos (N:M) para SQL.

### 4. Tratamiento de valores nulos

Se ha identificado un único registro sin artista, álbum ni nombre de canción.

Tras comprobar que no existía otra aparición con información completa en el dataset original, se ha excluido de ambas tablas de trabajo.

**Resultados posteriores:**

- Canciones: 89.740.
- Relaciones canción-género: 113.549.
- Valores nulos: 0.

### 5. Integridad referencial

Se ha comprobado que todas las relaciones canción-género apuntan a identificadores existentes en la tabla de canciones.

No se han detectado relaciones huérfanas antes de la exclusión del registro incompleto.

### 6. Validación de características musicales

Se han validado las siguientes variables:

`danceability`, `energy`, `speechiness`, `acousticness`, `instrumentalness`, `liveness` y `valence`.

Todas presentan valores dentro del intervalo esperado de 0 a 1.

También se han revisado los valores mínimos y máximos de popularidad, duración, tempo, volumen, tonalidad, modo y compás.

Se han identificado posibles valores atípicos que requieren evaluación posterior, especialmente en duración, tempo y compás.

### 7. Conclusión provisional

La limpieza inicial ha permitido consolidar las canciones, conservar sus asociaciones con géneros y eliminar un registro sin información descriptiva suficiente.

Las tablas resultantes constituyen una primera base para el modelo relacional del Music Recommender.

**Estado:** pasos 1-17 completados. Pendiente de finalizar la revisión de valores atípicos, realizar las validaciones finales y exportar los datos preparados para SQL.

In [19]:
# Paso 18: investigar posibles valores atípicos

print("Canciones con tempo = 0:", (tracks["tempo"] == 0).sum())

print(
    "Canciones con time_signature = 0:",
    (tracks["time_signature"] == 0).sum()
)

print(
    "Canciones de menos de 30 segundos:",
    (tracks["duration_ms"] < 30000).sum()
)

print(
    "Canciones de más de 30 minutos:",
    (tracks["duration_ms"] > 1800000).sum()
)

Canciones con tempo = 0: 157
Canciones con time_signature = 0: 162
Canciones de menos de 30 segundos: 14
Canciones de más de 30 minutos: 34


In [20]:
# Paso 19: comprobar coincidencias entre valores anómalos

tempo_cero = tracks["tempo"] == 0
compas_cero = tracks["time_signature"] == 0

print(
    "Canciones con ambos valores a cero:",
    (tempo_cero & compas_cero).sum()
)

print(
    "Solo tempo a cero:",
    (tempo_cero & ~compas_cero).sum()
)

print(
    "Solo compás a cero:",
    (~tempo_cero & compas_cero).sum()
)

Canciones con ambos valores a cero: 157
Solo tempo a cero: 0
Solo compás a cero: 5


In [21]:
# Paso 20: comprobar campos de texto vacíos

columnas_texto = ["artists", "album_name", "track_name"]

for columna in columnas_texto:
    vacios = tracks[columna].str.strip().eq("").sum()
    print(f"{columna}: {vacios} textos vacíos")

vacios_genero = track_genres["track_genre"].str.strip().eq("").sum()
print(f"track_genre: {vacios_genero} textos vacíos")

artists: 0 textos vacíos
album_name: 0 textos vacíos
track_name: 0 textos vacíos
track_genre: 0 textos vacíos


In [22]:
# Paso 21: validar la integridad de las tablas

duplicados_tracks = tracks["track_id"].duplicated().sum()

duplicados_relaciones = track_genres.duplicated(
    subset=["track_id", "track_genre"]
).sum()

relaciones_huerfanas = (
    ~track_genres["track_id"].isin(tracks["track_id"])
).sum()

print("Canciones duplicadas:", duplicados_tracks)
print("Relaciones duplicadas:", duplicados_relaciones)
print("Relaciones huérfanas:", relaciones_huerfanas)

Canciones duplicadas: 0
Relaciones duplicadas: 0
Relaciones huérfanas: 0


In [23]:
# Paso 22: revisar la estructura de las tablas

print("=== TRACKS ===")
print(tracks.dtypes.to_string())

print("\n=== TRACK_GENRES ===")
print(track_genres.dtypes.to_string())

=== TRACKS ===
track_id                str
artists                 str
album_name              str
track_name              str
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64

=== TRACK_GENRES ===
track_id       str
track_genre    str


In [24]:
# Paso 23: exportar las tablas preparadas

from pathlib import Path

ruta_procesados = Path("../data/processed")
ruta_procesados.mkdir(parents=True, exist_ok=True)

tracks.to_csv(
    ruta_procesados / "tracks_clean.csv",
    index=False,
    encoding="utf-8"
)

track_genres.to_csv(
    ruta_procesados / "track_genres_clean.csv",
    index=False,
    encoding="utf-8"
)

print("Archivos exportados correctamente:")
print(ruta_procesados / "tracks_clean.csv")
print(ruta_procesados / "track_genres_clean.csv")

Archivos exportados correctamente:
..\data\processed\tracks_clean.csv
..\data\processed\track_genres_clean.csv


## Resultados finales de limpieza | Pasos 18-23

### 1. Análisis de valores atípicos

Se han identificado:

- 157 canciones con `tempo = 0`.
- 162 canciones con `time_signature = 0`.
- 14 canciones con duración inferior a 30 segundos.
- 34 canciones con duración superior a 30 minutos.

Las 157 canciones con tempo cero también presentan compás cero. Otras cinco canciones tienen únicamente el compás a cero.

Se ha decidido conservar estos registros, ya que los valores detectados no demuestran por sí solos que sean errores. Las características rítmicas no determinadas podrán recibir un tratamiento específico durante el desarrollo del recomendador.

### 2. Validación de campos de texto

Se han revisado `artists`, `album_name`, `track_name` y `track_genre`.

No se han encontrado cadenas vacías ni valores compuestos únicamente por espacios.

### 3. Integridad de los datos

Las comprobaciones finales han confirmado:

- 0 identificadores de canción duplicados.
- 0 relaciones canción-género duplicadas.
- 0 relaciones huérfanas.

### 4. Revisión de estructura

Se han identificado los tipos de datos de las 19 columnas de `tracks` y las 2 columnas de `track_genres`.

Esta información servirá de referencia para seleccionar los tipos de datos y definir las restricciones en MySQL.

### 5. Exportación

Se han generado dos archivos CSV en `data/processed/`:

- `tracks_clean.csv`: 89.740 canciones.
- `track_genres_clean.csv`: 113.549 relaciones canción-género.

Estos archivos representan una primera versión limpia del dataset Spotify 114K y servirán como entrada para la siguiente fase de modelado relacional.

### 6. Conclusión

Se ha completado la fase inicial de limpieza y validación del dataset Spotify 114K mediante Python y Pandas.

**Siguiente fase:** definir los requisitos funcionales, identificar entidades y relaciones, establecer cardinalidades y construir el diagrama ERD de Music Recommender | SQL Edition antes de implementar la base de datos en MySQL.